# Handwritten Digit Classification

This notebook implements a simple handwritten digit classifier with these features:
1. Upload handwritten digit image
2. Automatic preprocessing (grayscale, resize, normalize)
3. Digit prediction for digits 0–9
4. Confidence score display
5. Prediction history stored using pandas and exported to CSV

In [1]:
import os
from datetime import datetime
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

try:
    import ipywidgets as widgets
    from ipywidgets import FileUpload
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

HISTORY_CSV = Path("prediction_history.csv")

digits = load_digits()
X = digits.data
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation="relu",
    solver="adam",
    batch_size=64,
    max_iter=50,
    random_state=42,
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Training complete.")
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Training complete.
Test accuracy: 0.9694444444444444

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.97      0.99        36
           1       0.92      0.92      0.92        36
           2       1.00      1.00      1.00        35
           3       1.00      1.00      1.00        37
           4       0.95      0.97      0.96        36
           5       0.97      1.00      0.99        37
           6       1.00      0.97      0.99        36
           7       0.97      1.00      0.99        36
           8       0.91      0.89      0.90        35
           9       0.97      0.97      0.97        36

    accuracy                           0.97       360
   macro avg       0.97      0.97      0.97       360
weighted avg       0.97      0.97      0.97       360



d:\DDP_calculate\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
def load_history():
    if HISTORY_CSV.exists():
        df = pd.read_csv(HISTORY_CSV, parse_dates=["timestamp"])
    else:
        df = pd.DataFrame(
            columns=["image_name", "predicted_digit", "confidence", "timestamp"]
        )
    return df

history_df = load_history()

RESAMPLE_FILTER = (
    Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.ANTIALIAS
)

def preprocess_image(source, target_size=(8, 8)):
    if isinstance(source, (str, Path)):
        image = Image.open(source)
    elif isinstance(source, BytesIO):
        image = Image.open(source)
    else:
        image = source

    image = image.convert("L")
    image = image.resize(target_size, RESAMPLE_FILTER)
    image_array = np.array(image, dtype=np.float32)

    if image_array.mean() > 127:
        image_array = 255 - image_array

    image_array = image_array / 255.0 * 16.0
    return image_array.flatten()


def predict_image(source, image_name="uploaded_image"):
    image_array = preprocess_image(source)
    X_input = scaler.transform(image_array.reshape(1, -1))
    probabilities = model.predict_proba(X_input)[0]
    digit = int(np.argmax(probabilities))
    confidence = float(np.max(probabilities))
    return digit, confidence


def add_history(image_name, digit, confidence):
    global history_df
    record = {
        "image_name": image_name,
        "predicted_digit": digit,
        "confidence": f"{confidence * 100:.1f}%",
        "timestamp": datetime.now(),
    }
    history_df = pd.concat([history_df, pd.DataFrame([record])], ignore_index=True)
    history_df.to_csv(HISTORY_CSV, index=False)
    return history_df

In [ ]:
if WIDGETS_AVAILABLE:
    upload_widget = FileUpload(accept="image/*", multiple=False)
    display(upload_widget)
    print("Upload a handwritten digit image using the widget above, then run the next cell.")
else:
    print("ipywidgets is not available. Use the file path input method in the next cell.")

FileUpload(value=(), accept='image/*', description='Upload')

Upload a handwritten digit image using the widget above, then run the next cell.


In [6]:
def get_upload_source():
    if WIDGETS_AVAILABLE and upload_widget.value:
        name = list(upload_widget.value.keys())[0]
        content = upload_widget.value[name]["content"]
        return name, BytesIO(content)

    image_path = input("Enter local image path for a handwritten digit image: ").strip()
    return Path(image_path).name, image_path

image_name, source = get_upload_source()
prediction, confidence = predict_image(source, image_name=image_name)
print(f"Digit {prediction} — {confidence * 100:.1f}% confidence")
history_df = add_history(image_name, prediction, confidence)
display(history_df.tail(10))

FileNotFoundError: [Errno 2] No such file or directory: 'w'